[\![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joyalissa13/QML/blob/main/qml_pennylane_intro.ipynb)

# Introduction to PennyLane — Training a Variational Quantum Classifier

## What is PennyLane?

PennyLane lets you write quantum circuits as Python functions and treat them like any other
differentiable function — you can compute gradients through them with autograd, PyTorch,
TensorFlow, or JAX, exactly like you would with a neural network layer. You build a *variational circuit* (a circuit with
trainable parameters), define a cost function, and optimize the parameters with gradient descent (or other methods),
just like training a classical neural net.


In [1]:
%pip install pennylane scikit-learn matplotlib

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 37.7 MB/s  0:00:00 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pennylane as qml
from pennylane import numpy as np  # PennyLane's autograd-wrapped numpy — use this instead of plain numpy
import matplotlib.pyplot as plt

print("PennyLane version:", qml.__version__)

PennyLane version: 0.45.1


## Part 1 — Quantum Circuits

Ccreating a device, writing a circuit, and measuring it:

When we *measure*, we typically don't collapse to a single bit outcome — we ask for the **expectation value** of an observable $\hat{O}$, given by the Born rule:

$$\langle \hat{O} \rangle = \langle \psi | \hat{O} | \psi \rangle$$

For the Pauli-Z observable acting on a qubit in state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$, this becomes

$$\langle Z \rangle = |\alpha|^2 - |\beta|^2 \in [-1, 1],$$

i.e. the difference between the probabilities of measuring $|0\rangle$ and $|1\rangle$. Unlike a raw bit-string sample, $\langle Z \rangle$ is a **smooth, real-valued function of the circuit's parameters** — since $\alpha$ and $\beta$ themselves vary smoothly with gate angles like $\theta$ in $R_X(\theta)$. This differentiability is exactly what lets us backpropagate through a quantum circuit and treat it as a trainable layer in an ML pipeline.


In [3]:
# A device with 1 qubit ("wire"), simulated exactly (no shot noise)
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def simple_circuit(theta):
    qml.RX(theta, wires=0)   # rotate around the X axis by angle theta
    return qml.expval(qml.PauliZ(0))  # measure expectation of Z

for theta in [0, np.pi/4, np.pi/2, np.pi]:
    print(f"theta={theta:.3f} -> <Z> = {simple_circuit(theta):.3f}")

theta=0.000 -> <Z> = 1.000
theta=0.785 -> <Z> = 0.707
theta=1.571 -> <Z> = 0.000
theta=3.142 -> <Z> = -1.000


Notice the output smoothly goes from $+1$ (no rotation) to $-1$ (rotation by $\pi$, flipping
the qubit to $|1\rangle$). Because this is differentiable, we can ask PennyLane for the gradient
directly: 

In [4]:
grad_fn = qml.grad(simple_circuit)
print("d<Z>/dtheta at theta=pi/4:", grad_fn(np.array(np.pi/4, requires_grad=True)))

d<Z>/dtheta at theta=pi/4: -0.7071067811865476


### Simple functions

Create a device with **2 wires** and a QNode that:
1. Applies `qml.Hadamard` to wire 0
2. Applies `qml.CNOT` with control wire 0 and target wire 1 (this creates entanglement!)
3. Applies `qml.RY(phi, wires=1)`
4. Returns `qml.expval(qml.PauliZ(1))`

Then evaluate it at `phi = 0.3`.

(Fill in the `# TODO` lines below. Solutions for every exercise are in the **Solutions** section
at the very end of the notebook.)


In [6]:
dev2 = qml.device("default.qubit", wires=2)
# wires mean the number of qubits in the circuit. In this case, we have 2 qubits.
# wires = [0, 1] first 0 and sec 1

@qml.qnode(dev2)
def exercise_1_circuit(phi):
    # TODO: Hadamard on wire 0
    qml.Hadamard(wires=0)
    # TODO: CNOT(wires=[0, 1])
    qml.CNOT(wires=[0, 1])
    # TODO: RY(phi, wires=1)
    qml.RY(phi, wires=1)
    # TODO: return qml.expval(qml.PauliZ(1))
    return qml.expval(qml.PauliZ(1))

result = exercise_1_circuit(0.3)
print(result)

0.0


## The Dataset

We'll use the classic **Iris** flower dataset (built into scikit-learn — no manual download
needed, it ships with the library and loads instantly and offline). It has 150 samples, 4
features (sepal/petal length & width), and 3 classes. We'll simplify to a **binary classification
problem** (`setosa` vs `versicolor` [these are iris flowers]) using the first 2 features so we can also visualize the
decision boundary in 2D, and so 2 qubits (with angle embedding) suffice.

> If you want, you can get the raw CSV - just run `load_iris(as_frame=True).frame.to_csv("iris.csv")` after the
> next cell and it'll be saved to your working directory (or just google it)


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

iris = load_iris()
X_all, y_all = iris.data, iris.target
print("Features:", iris.feature_names) # overview of the features

# Keep it binary (classes 0 and 1) and only the first 2 features for easy 2D visualization
mask = y_all < 2 # ignore other calsses - only keep setosa and versicolor
X = X_all[mask][:, :2] # here we only keep the first 2 features (sepal length and sepal width) - you can see the description of the features in iris.feature_names
y = y_all[mask]

# Rescale features to [0, pi] so they map nicely onto rotation-gate angles
scaler = MinMaxScaler(feature_range=(0, np.pi))

# the reason why we scale is because the rotation gates in quantum circuits take angles as input, and we want to ensure that our features are within a suitable range for these gates. 
# By scaling the features to [0, pi], we can effectively use them as rotation angles in our quantum circuit.
# In ML we scale features (e.g. to [0,1] or min/max) because models that rely on distances
# or gradients can be skewed by features with larger raw ranges dominating others leading to poor performance. Scaling ensures that all features contribute equally to the model's learning process.

X = scaler.fit_transform(X)

# Map labels {0, 1} -> {-1, +1} to match the <Z> expectation value range
y = np.where(y == 0, -1, 1)

# Split into train/test sets (there are several possible approachese to splitting - feel free to explore other options and check how they affect the model performance :D)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Train samples:", len(X_train), " Test samples:", len(X_test))
print("Feature range:", X.min(), "to", X.max())

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=(y == 1), cmap="coolwarm", edgecolors="k")
plt.xlabel("sepal length (scaled)")
plt.ylabel("sepal width (scaled)")
plt.title("Iris: setosa (blue) vs versicolor (red)")
plt.tight_layout()
plt.show()

## Encoding Classical Data into a Quantum Circuit

To feed classical features into a quantum circuit we need a **feature map / embedding**. A simple
and common choice is **angle embedding**: encode each feature as a rotation angle on its own
qubit, e.g. `qml.RX(x[i], wires=i)`. PennyLane has this built in as `qml.AngleEmbedding`, but
let's first write it by hand so the mechanics are clear.

### 🧩 Exercise 2 — Write an angle-embedding feature map

Complete the function below: for a 2-feature input `x`, apply `qml.RX(x[0], wires=0)` and
`qml.RX(x[1], wires=1)`.


In [ ]:
def feature_map(x):
    # TODO: qml.RX(x[0], wires=0)
    # TODO: qml.RX(x[1], wires=1)
    pass

## Part 4 — The Variational Ansatz

After encoding the data, we apply a **variational ansatz**: a layer (or several layers) of
trainable rotation gates plus entangling gates. This is the "trainable neural-network-like" part
of the circuit — the parameters get optimized during training, while the feature map above always
stays fixed for a given input.

A minimal one-layer ansatz for 2 qubits:
1. `qml.RY(weights[0], wires=0)`
2. `qml.RY(weights[1], wires=1)`
3. `qml.CNOT(wires=[0, 1])` — entangle the qubits so the model can learn correlations between features

### 🧩 Exercise 3 — Write the variational ansatz

Complete `variational_layer(weights)` below, where `weights` is an array of length 2.


In [ ]:
def variational_layer(weights):
    # TODO: qml.RY(weights[0], wires=0)
    # TODO: qml.RY(weights[1], wires=1)
    # TODO: qml.CNOT(wires=[0, 1])
    pass

## Part 5 — Assembling the Full Model

Now combine `feature_map` + `variational_layer` (we'll stack a few layers for more expressive
power) into a single QNode, and read out the prediction as `qml.expval(qml.PauliZ(0))` — a
continuous value in $[-1, 1]$ we'll compare against our $\pm 1$ labels.


In [ ]:
n_qubits = 2
n_layers = 3
dev_vqc = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_vqc)
def circuit(weights, x):
    qml.RX(x[0], wires=0)
    qml.RX(x[1], wires=1)
    for layer in range(n_layers):
        qml.RY(weights[layer, 0], wires=0)
        qml.RY(weights[layer, 1], wires=1)
        qml.CNOT(wires=[0, 1])
    return qml.expval(qml.PauliZ(0))

def variational_classifier(weights, bias, x):
    return circuit(weights, x) + bias

# Quick sanity check with random weights
init_weights = 0.01 * np.random.randn(n_layers, n_qubits, requires_grad=True)
init_bias = np.array(0.0, requires_grad=True)
print("Sample prediction:", variational_classifier(init_weights, init_bias, X_train[0]))
print(qml.draw(circuit)(init_weights, X_train[0]))

## Part 6 — Cost Function

We'll use **mean squared error** between the circuit's continuous output and the $\pm 1$ labels
— simple, and it works well for this kind of expectation-value readout.

### 🧩 Exercise 4 — Write the cost function

Complete `cost(weights, bias, X, y)`:
1. Compute predictions for every sample in `X` using `variational_classifier`
2. Return the mean squared error against `y`

Hint: `np.mean((predictions - y) ** 2)`


In [ ]:
def cost(weights, bias, X, y):
    # TODO: predictions = [variational_classifier(weights, bias, x) for x in X]
    # TODO: predictions = np.stack(predictions)
    # TODO: return np.mean((predictions - y) ** 2)
    pass

def accuracy(weights, bias, X, y):
    predictions = np.sign([variational_classifier(weights, bias, x) for x in X])
    return np.mean(predictions == y)

## Part 7 — Training Loop

Standard gradient-descent training loop: pick an optimizer (PennyLane's `NesterovMomentumOptimizer`
works nicely here), loop over epochs, call `opt.step()` with the cost function.

### 🧩 Exercise 5 — Write the training loop

Complete the loop body:
1. Call `opt.step(lambda w, b: cost(w, b, X_train, y_train), weights, bias)` — this returns the
   **updated** `(weights, bias)`
2. Every 10 steps, print the cost and train/test accuracy


In [ ]:
opt = qml.NesterovMomentumOptimizer(stepsize=0.3)
weights = 0.01 * np.random.randn(n_layers, n_qubits, requires_grad=True)
bias = np.array(0.0, requires_grad=True)

n_epochs = 40
cost_history = []

for epoch in range(n_epochs):
    # TODO: weights, bias = opt.step(lambda w, b: cost(w, b, X_train, y_train), weights, bias)
    current_cost = cost(weights, bias, X_train, y_train)
    cost_history.append(current_cost)
    if epoch % 10 == 0:
        train_acc = accuracy(weights, bias, X_train, y_train)
        test_acc = accuracy(weights, bias, X_test, y_test)
        print(f"Epoch {epoch:3d} | cost {current_cost:.4f} | train acc {train_acc:.2f} | test acc {test_acc:.2f}")

Once your training loop above is filled in and run, plot the loss curve to see it converge:

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(cost_history)
plt.xlabel("Epoch")
plt.ylabel("Cost (MSE)")
plt.title("Training loss")
plt.tight_layout()
plt.show()

print("Final test accuracy:", accuracy(weights, bias, X_test, y_test))

## Part 8 — Visualize the Decision Boundary

Because we used only 2 features, we can plot the learned decision boundary directly.


In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min(), X[:, 0].max(), 30),
    np.linspace(X[:, 1].min(), X[:, 1].max(), 30),
)
grid = np.c_[xx.ravel(), yy.ravel()]
preds = np.array([variational_classifier(weights, bias, point) for point in grid])
preds = preds.reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, preds, levels=20, cmap="coolwarm", alpha=0.6)
plt.scatter(X[:, 0], X[:, 1], c=(y == 1), cmap="coolwarm", edgecolors="k")
plt.xlabel("sepal length (scaled)")
plt.ylabel("sepal width (scaled)")
plt.title("Learned decision boundary")
plt.tight_layout()
plt.show()

## 🧩 Bonus Exercises

Try any of these to deepen your understanding — no solutions provided, experiment freely!

1. **More layers**: increase `n_layers` to 5–6. Does accuracy improve? Does training get slower
   or less stable?
2. **Built-in templates**: replace the hand-written feature map/ansatz with
   `qml.AngleEmbedding(x, wires=range(n_qubits))` and
   `qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))`. Check `qml.draw` to compare
   the circuit structure.
3. **All 3 Iris classes**: extend to 3-class classification using `qml.expval(qml.PauliZ(0))` and
   `qml.expval(qml.PauliZ(1))` together (2 outputs), or a one-vs-rest approach.
4. **Different optimizer**: try `qml.AdamOptimizer` or `qml.GradientDescentOptimizer` and compare
   convergence speed.
5. **Shot noise**: create a device with `qml.device("default.qubit", wires=n_qubits, shots=100)`
   to simulate a real quantum computer's measurement noise, and see how training is affected.
6. **All 4 features**: use all 4 Iris features with 4 qubits instead of 2.


---
## Solutions

Only look here after attempting the exercises above! Each solution corresponds to the exercise
with the same number.


### Solution — Exercise 1

In [ ]:
dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2)
def exercise_1_solution(phi):
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    qml.RY(phi, wires=1)
    return qml.expval(qml.PauliZ(1))

print(exercise_1_solution(0.3))

### Solution — Exercise 2

In [ ]:
def feature_map_solution(x):
    qml.RX(x[0], wires=0)
    qml.RX(x[1], wires=1)

### Solution — Exercise 3

In [ ]:
def variational_layer_solution(weights):
    qml.RY(weights[0], wires=0)
    qml.RY(weights[1], wires=1)
    qml.CNOT(wires=[0, 1])

### Solution — Exercise 4

In [ ]:
def cost_solution(weights, bias, X, y):
    predictions = np.stack([variational_classifier(weights, bias, x) for x in X])
    return np.mean((predictions - y) ** 2)

### Solution — Exercise 5

In [ ]:
opt = qml.NesterovMomentumOptimizer(stepsize=0.3)
weights_sol = 0.01 * np.random.randn(n_layers, n_qubits, requires_grad=True)
bias_sol = np.array(0.0, requires_grad=True)

for epoch in range(n_epochs):
    weights_sol, bias_sol = opt.step(
        lambda w, b: cost(w, b, X_train, y_train), weights_sol, bias_sol
    )
    if epoch % 10 == 0:
        train_acc = accuracy(weights_sol, bias_sol, X_train, y_train)
        test_acc = accuracy(weights_sol, bias_sol, X_test, y_test)
        print(f"Epoch {epoch:3d} | cost {cost(weights_sol, bias_sol, X_train, y_train):.4f} "
              f"| train acc {train_acc:.2f} | test acc {test_acc:.2f}")

print("Final test accuracy:", accuracy(weights_sol, bias_sol, X_test, y_test))

## Where to go next

- [PennyLane demos](https://pennylane.ai/qml/demonstrations) — dozens of worked QML tutorials
- [PennyLane docs: templates](https://docs.pennylane.ai/en/stable/introduction/templates.html) —
  pre-built embeddings and ansätze like `AngleEmbedding`, `AmplitudeEmbedding`,
  `StronglyEntanglingLayers`, `BasicEntanglerLayers`
- Try swapping `default.qubit` for a noisy simulator or real hardware backend (e.g. via
  PennyLane-Qiskit or Amazon Braket plugins) once you're comfortable with the basics
